# 11. Baseball-Specific Model Insights

## Goals

Translate the trained model's predictions and interpretability outputs into baseball-relevant conclusions.

This notebook does not ask whether the model predicts injury. Notebook 10 covers that.
It asks **what the model suggests about pitcher health, usage, role design, workload management,
pitch mix construction, and possible future strategy changes**.

Key questions:
- What did the model learn about pitcher health?
- Which baseball behaviors appear most risky?
- Are there pitcher archetypes with different health profiles?
- Are there nonlinear danger zones in velocity, pitch mix, workload, rest, or role?
- What insights might lead to new pitcher usage strategies?

> **Methodology note:** This notebook uses associative language throughout ("associated with",
> "model assigns higher risk to", "suggests"). Do not interpret patterns here as causal unless
> a formal causal inference design is applied downstream.

---

## Inputs

- `data/processed/feature_matrix.parquet`
- `data/processed/injury_risk_plus_scores.parquet`
- `data/processed/pitcher_archetypes.parquet` (if available)
- `models/multitask_chained.joblib`
- `models/baseline_xgboost.joblib`
- SHAP outputs from Notebook 10 (if available)

## Outputs

**Figures** saved to `reports/figures/`:
- `risk_vs_slider_usage.png`
- `risk_vs_velocity.png`
- `risk_vs_workload.png`
- `risk_vs_rest_days.png`
- `risk_heatmap_velocity_slider.png`
- `risk_heatmap_velocity_workload.png`
- `risk_heatmap_slider_rest.png`
- `risk_by_pitcher_archetype.png`
- `risk_by_role.png`
- `performance_risk_frontier.png`
- `pre_injury_risk_trajectory.png`

**Tables** saved to `reports/tables/`:
- `baseball_specific_insights_summary.csv`

In [ ]:
import warnings, json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.ndimage import gaussian_filter
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

# Plotting style
CLR_LOW   = '#2166ac'   # blue  — low risk
CLR_HIGH  = '#d73027'   # red   — high risk
CLR_MID   = '#f4a582'   # light orange — moderate risk
CLR_NEUT  = '#636363'
PALETTE   = sns.color_palette('RdYlBu_r', 10)
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

FIGURES_DIR = Path('reports/figures')
TABLES_DIR  = Path('reports/tables')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')
print('Figures →', FIGURES_DIR.resolve())

---
## Section 1: Load Data and Merge Risk Scores

Load the feature matrix and Injury Risk+ scores, then merge into a single analysis dataframe.
Validate row counts, missing Risk+ values, key identifier columns, and label coverage.

In [ ]:
# Load feature matrix
fm_path = Path('data/processed/feature_matrix.parquet')
if not fm_path.exists():
    raise FileNotFoundError('Run notebooks 01-05 first to build feature_matrix.parquet')
fm = pd.read_parquet(fm_path)
fm['game_date'] = pd.to_datetime(fm['game_date'])
print(f'Feature matrix  : {fm.shape[0]:,} rows × {fm.shape[1]} cols')

# Load Injury Risk+ scores
# NB09 saves one row per pitcher-season (not per game appearance).
# Merge on pitcher + season so every game row picks up its season-level Risk+.
risk_path = Path('data/processed/injury_risk_plus_scores.parquet')
analysis = fm.copy()
if risk_path.exists():
    risk = pd.read_parquet(risk_path)
    risk_merge = (
        risk[['pitcher_id', 'season', 'injury_risk_plus']]
        .rename(columns={'pitcher_id': 'pitcher'})
    )
    analysis = analysis.merge(risk_merge, on=['pitcher', 'season'], how='left')
    n_matched = analysis['injury_risk_plus'].notna().sum()
    print(f'Risk+ scores    : {len(risk):,} pitcher-seasons → {n_matched:,} game rows matched')
else:
    print('WARNING: injury_risk_plus_scores.parquet not found — run notebook 09 first.')
    print('Falling back to injured_next_30d label as a risk proxy.')
    fallback = fm.get('injured_next_30d', pd.Series(0, index=fm.index))
    analysis['injury_risk_plus'] = (fallback * 100 + 100).clip(0, 300)

# Load pitcher archetypes (optional)
arch_path = Path('data/processed/pitcher_archetypes.parquet')
archetypes = pd.read_parquet(arch_path) if arch_path.exists() else None
print(f'Archetypes      : {"loaded" if archetypes is not None else "not found — will derive in Section 8"}')

print(f'\nAnalysis frame  : {analysis.shape[0]:,} rows × {analysis.shape[1]} cols')
print(f'Risk+ coverage  : {analysis["injury_risk_plus"].notna().mean():.1%}')
print(f'Pitchers        : {analysis["pitcher"].nunique():,}')
print(f'Date range      : {analysis["game_date"].min().date()} → {analysis["game_date"].max().date()}')
if 'injured_next_30d' in analysis.columns:
    print(f'Injury label (30d): {analysis["injured_next_30d"].mean():.1%} positive rate')

---
## Section 2: Risk+ Distribution

Characterize the league-wide distribution of Injury Risk+ scores.

**Research questions:**
- What does league-average risk look like?
- How many pitchers are above 120, 150, and 200 Risk+?
- Are extreme-risk pitchers rare or common?

In [ ]:
# Risk+ distribution visualizations
risk_vals = analysis['injury_risk_plus'].dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Histogram
axes[0].hist(risk_vals, bins=40, color=CLR_NEUT, alpha=0.8, edgecolor='white')
axes[0].axvline(100, color='black', ls='--', lw=2, label='League avg (100)')
for thresh, col in [(120, CLR_MID), (150, CLR_HIGH)]:
    axes[0].axvline(thresh, color=col, ls=':', lw=1.5, label=f'Risk+ {thresh}')
axes[0].set_xlabel('Injury Risk+')
axes[0].set_ylabel('Count')
axes[0].set_title('Injury Risk+ Distribution', fontweight='bold')
axes[0].legend(fontsize=9)

# 2) KDE density
from scipy.stats import gaussian_kde
kde = gaussian_kde(risk_vals.dropna())
x_grid = np.linspace(risk_vals.min(), risk_vals.max(), 300)
axes[1].fill_between(x_grid, kde(x_grid), alpha=0.4, color=CLR_NEUT)
axes[1].plot(x_grid, kde(x_grid), color=CLR_NEUT, lw=2)
axes[1].axvline(100, color='black', ls='--', lw=2)
axes[1].set_xlabel('Injury Risk+')
axes[1].set_ylabel('Density')
axes[1].set_title('Risk+ Density', fontweight='bold')

# 3) Percentile table
pcts = [10, 25, 50, 75, 90, 95, 99]
pct_vals = np.percentile(risk_vals.dropna(), pcts)
axes[2].barh([f'P{p}' for p in pcts], pct_vals, color=
             [CLR_LOW if v < 100 else (CLR_HIGH if v > 130 else CLR_NEUT) for v in pct_vals])
axes[2].axvline(100, color='black', ls='--', lw=1.5)
axes[2].set_xlabel('Injury Risk+')
axes[2].set_title('Risk+ Percentiles', fontweight='bold')

plt.suptitle('Figure 1: Injury Risk+ Distribution', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'risk_plus_distribution.png')
plt.show()

# Threshold summary
print('Threshold analysis:')
for thresh in [120, 150, 175, 200]:
    n = (risk_vals > thresh).sum()
    pct = (risk_vals > thresh).mean()
    print(f'  Risk+ > {thresh}: {n:,} pitcher-games ({pct:.1%})')

---
## Section 3: Risk+ vs Pitch Mix

Examine how pitch mix composition is associated with Injury Risk+.

**Research questions:**
- Does risk increase above certain slider usage thresholds?
- Are breaking-ball-heavy pitchers riskier?
- Are there safer pitch mix profiles?

In [ ]:
# Risk+ vs pitch mix features
mix_features = {
    'sl_pct':       'Slider Usage %',
    'breaking_pct': 'Breaking Ball %',
    'fb_pct':       'Fastball %',
    'ch_pct':       'Changeup %',
}
mix_features = {k: v for k, v in mix_features.items() if k in analysis.columns}

fig, axes = plt.subplots(1, len(mix_features), figsize=(5 * len(mix_features), 5))
if len(mix_features) == 1:
    axes = [axes]

for ax, (feat, label) in zip(axes, mix_features.items()):
    df = analysis[['injury_risk_plus', feat]].dropna()
    ax.scatter(df[feat] * 100, df['injury_risk_plus'],
               alpha=0.3, s=12, color=CLR_NEUT)
    # Smoothed trend (bin averages)
    bins = pd.cut(df[feat], bins=10)
    trend = df.groupby(bins, observed=True)['injury_risk_plus'].mean()
    bin_centers = [interval.mid for interval in trend.index]
    ax.plot([x * 100 for x in bin_centers], trend.values,
            color=CLR_HIGH, lw=2.5, label='Bin average')
    ax.axhline(100, color='black', ls='--', lw=1, alpha=0.5)
    ax.set_xlabel(f'{label} (%)')
    ax.set_ylabel('Injury Risk+' if ax == axes[0] else '')
    ax.set_title(f'Risk+ vs {label}', fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Figure 2: Risk+ vs Pitch Mix', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'risk_vs_slider_usage.png')
plt.show()

# Decile analysis for slider usage
if 'sl_pct' in analysis.columns:
    df_sl = analysis[['injury_risk_plus', 'sl_pct']].dropna()
    all_labels = ['Q1\n(low)', 'Q2', 'Q3', 'Q4', 'Q5\n(high)']
    _, _bins = pd.qcut(df_sl['sl_pct'], q=5, retbins=True, duplicates='drop')
    n_actual = len(_bins) - 1
    df_sl['sl_decile'] = pd.qcut(df_sl['sl_pct'], q=5, labels=all_labels[:n_actual], duplicates='drop')
    print('Average Risk+ by slider usage quintile:')
    print(df_sl.groupby('sl_decile', observed=True)['injury_risk_plus'].agg(['mean', 'median', 'count']).round(1))

### Research Note: Slider Usage vs UCL Injury Type Specificity

The scatter plots above show a positive association between slider usage % and overall Injury Risk+,
consistent with Tanaka et al. 2024 (PMC 11369970) which found slider utilization as a top SHAP
feature for shoulder/elbow injury in MLB pitchers.

Two critical qualifications from the literature:

1. **Velocity ≠ usage as the mechanism.** Fleisig et al. 2016 (*J Shoulder Elbow Surgery*)
   found **no significant difference in slider velocity** between pitchers who underwent UCL
   reconstruction and matched controls. The mechanical risk comes from slider *usage rate*
   (repeated forearm pronation and varus torque), not slider velocity.

2. **Injury type specificity.** Slider utilization predicts **elbow/UCL injuries** specifically.
   Because our Injury Risk+ predicts *all injury types combined*, the slider signal is diluted.
   A slider-heavy pitcher whose non-slider pitches are mechanically safe will still appear elevated.
   Isolating elbow-specific injuries would sharpen this signal significantly.

**Practical implication for Risk+ interpretation:** An elevated slider % contribution to a
pitcher's Risk+ score should be read as an elbow-stress indicator specifically, not as a
general health concern across all injury categories.

---
## Section 4: Risk+ vs Velocity

Examine velocity features as injury risk predictors.

**Research questions:**
- Is high velocity itself risky, or is it velocity change that matters?
- Are velocity spikes riskier than sustained velocity?
- Does velocity decline before injury appear meaningful?

In [ ]:
# Risk+ vs velocity features
velo_features = {
    'fb_velo_mean':        'Mean FB Velocity (mph)',
    'fb_velo_max':         'Max FB Velocity (mph)',
    'velo_change_7_30d':   'Velocity Change (7-day vs 30-day)',
    'velo_delta_vs_season':'Velocity vs Season Avg',
}
velo_features = {k: v for k, v in velo_features.items() if k in analysis.columns}

if not velo_features:
    print('Velocity delta features not yet computed — run notebook 05 with full data.')
else:
    fig, axes = plt.subplots(1, len(velo_features), figsize=(5 * len(velo_features), 5))
    if len(velo_features) == 1:
        axes = [axes]

    for ax, (feat, label) in zip(axes, velo_features.items()):
        df = analysis[['injury_risk_plus', feat]].dropna()
        ax.scatter(df[feat], df['injury_risk_plus'], alpha=0.3, s=12, color=CLR_NEUT)
        bins = pd.cut(df[feat], bins=10)
        trend = df.groupby(bins, observed=True)['injury_risk_plus'].mean()
        bin_centers = [interval.mid for interval in trend.index]
        ax.plot(bin_centers, trend.values, color=CLR_HIGH, lw=2.5)
        ax.axhline(100, color='black', ls='--', lw=1, alpha=0.5)
        ax.set_xlabel(label)
        ax.set_ylabel('Injury Risk+' if ax == axes[0] else '')
        ax.set_title(f'Risk+ vs {label.split("(")[0].strip()}', fontweight='bold')

    plt.suptitle('Figure 3: Risk+ vs Velocity', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'risk_vs_velocity.png')
    plt.show()

### 4b. Velocity Decline Threshold Analysis

The scatter plots above show the marginal association between velocity features and Risk+.
A key literature question is: **at what decline magnitude does risk become clinically significant?**

PMC 12717397 (2025) found that acute UCL failure is characterized by an **abrupt** velocity
drop (>1.5 SD below baseline) *on the injury pitch itself*, not a gradual multi-outing
decline. This means our `velo_delta_vs_season` feature captures **chronic fatigue accumulation**
(week-to-week drift from the season mean), which is a different mechanism than the acute
decompensation signal. Both matter, but in different ways:

- **Chronic drift** (captured here): pitcher is systematically below their season baseline,
  indicating cumulative fatigue or biomechanical accommodation
- **Acute decompensation** (NOT captured): a single-outing sharp drop, visible only in
  pitch-by-pitch data

The binned analysis below tests whether chronic drift categories show a dose-response
relationship with model-predicted Risk+.

In [ ]:
# Test whether chronic season-average velocity decline shows a dose-response with predicted Risk+.
if 'velo_delta_vs_season' in analysis.columns:
    df_vd = analysis[['injury_risk_plus', 'velo_delta_vs_season', 'injured_next_30d']].dropna()
    vd_std = df_vd['velo_delta_vs_season'].std()
    print(f'velo_delta_vs_season: mean={df_vd["velo_delta_vs_season"].mean():.2f} mph, '
          f'std={vd_std:.2f} mph')

    df_vd['velo_decline_cat'] = pd.cut(
        df_vd['velo_delta_vs_season'],
        bins=[-np.inf, -2.0, -1.0, 0.0, np.inf],
        labels=['>2 mph decline', '1–2 mph decline', '0–1 mph decline', 'At/above season avg'],
    )

    summary = df_vd.groupby('velo_decline_cat', observed=True).agg(
        mean_risk_plus   = ('injury_risk_plus', 'mean'),
        median_risk_plus = ('injury_risk_plus', 'median'),
        injury_rate      = ('injured_next_30d', 'mean'),
        n                = ('injury_risk_plus', 'count'),
    ).round(2)
    summary['injury_rate_pct'] = (summary['injury_rate'] * 100).round(1)

    print('\nVelocity decline → Risk+ dose-response:')
    print(summary[['mean_risk_plus', 'median_risk_plus', 'injury_rate_pct', 'n']].to_string())
    mph_per_sd = vd_std
    print(f'\n1 mph = {1/mph_per_sd:.2f} SD  |  PMC 12717397 acute threshold: 1.5 SD on injury pitch')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    cats = summary.index
    axes[0].bar(cats, summary['mean_risk_plus'],
                color=[CLR_HIGH if v > 105 else (CLR_LOW if v < 98 else CLR_NEUT)
                       for v in summary['mean_risk_plus']])
    axes[0].axhline(100, color='black', ls='--', lw=1.5)
    axes[0].set_ylabel('Avg Injury Risk+')
    axes[0].set_title('Risk+ by Velocity Decline Category', fontweight='bold')
    axes[0].tick_params(axis='x', rotation=20)

    axes[1].bar(cats, summary['injury_rate_pct'],
                color=[CLR_HIGH if v > summary['injury_rate_pct'].mean() else CLR_LOW
                       for v in summary['injury_rate_pct']])
    axes[1].axhline(summary['injury_rate_pct'].mean(), color='black', ls='--', lw=1.5,
                    label='Overall avg')
    axes[1].set_ylabel('30-Day Injury Rate (%)')
    axes[1].set_title('Injury Rate by Velocity Decline Category', fontweight='bold')
    axes[1].tick_params(axis='x', rotation=20)
    axes[1].legend(fontsize=9)

    plt.suptitle('Figure 3b: Velocity Decline Threshold Analysis',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'velo_decline_threshold.png')
    plt.show()
    print(f'Saved {FIGURES_DIR}/velo_decline_threshold.png')
else:
    print('velo_delta_vs_season not available — skipping velocity decline threshold analysis.')

---
## Section 5: Risk+ vs Workload

Examine workload features and their association with predicted risk.

**Research questions:**
- Are there clear workload danger zones?
- Does risk accelerate above certain pitch counts?
- Is ACWR useful for pitcher injury risk?

In [ ]:
# Risk+ vs workload features
workload_features = {
    'pitch_count':   'Pitches This Outing',
    'pitches_7d':    'Pitches Last 7 Days',
    'pitches_28d':   'Pitches Last 28 Days',
    'acwr_7_28':     'ACWR (7-day / 28-day)',
}
workload_features = {k: v for k, v in workload_features.items() if k in analysis.columns}

fig, axes = plt.subplots(1, len(workload_features), figsize=(5 * len(workload_features), 5))
if len(workload_features) == 1:
    axes = [axes]

for ax, (feat, label) in zip(axes, workload_features.items()):
    df = analysis[['injury_risk_plus', feat]].dropna()
    ax.scatter(df[feat], df['injury_risk_plus'], alpha=0.25, s=10, color=CLR_NEUT)
    bins = pd.cut(df[feat], bins=12)
    trend = df.groupby(bins, observed=True)['injury_risk_plus'].mean()
    bin_centers = [interval.mid for interval in trend.index]
    ax.plot(bin_centers, trend.values, color=CLR_HIGH, lw=2.5)
    ax.axhline(100, color='black', ls='--', lw=1, alpha=0.5)
    ax.set_xlabel(label)
    ax.set_ylabel('Injury Risk+' if ax == axes[0] else '')
    ax.set_title(f'Risk+ vs {label}', fontweight='bold')

plt.suptitle('Figure 4: Risk+ vs Workload', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'risk_vs_workload.png')
plt.show()

# ACWR threshold analysis
if 'acwr_7_28' in analysis.columns:
    df_aw = analysis[['injury_risk_plus', 'acwr_7_28']].dropna()
    print('Average Risk+ by ACWR zone:')
    df_aw['acwr_zone'] = pd.cut(df_aw['acwr_7_28'],
                                 bins=[-np.inf, 0.8, 1.0, 1.3, 1.5, np.inf],
                                 labels=['<0.8', '0.8–1.0', '1.0–1.3', '1.3–1.5', '>1.5'])
    print(df_aw.groupby('acwr_zone', observed=True)['injury_risk_plus']
               .agg(['mean', 'count']).round(1))

---
## Section 6: Risk+ vs Rest and Recovery

Examine how rest and recovery patterns relate to predicted injury risk.

**Research questions:**
- Does extra rest lower predicted risk?
- Is short rest only risky for certain pitcher types?
- Do relievers show different rest-risk patterns than starters?

In [ ]:
# Risk+ vs rest and recovery features
if 'days_rest' not in analysis.columns:
    print('days_rest not in feature matrix — run notebook 05 with full data.')
else:
    df_rest = analysis[['injury_risk_plus', 'days_rest', 'pitch_count']].dropna()
    # Role proxy from pitch count
    df_rest['role'] = np.where(df_rest['pitch_count'] >= 60, 'Starter', 'Reliever')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1) Overall Risk+ by days rest
    rest_trend = df_rest.groupby('days_rest')['injury_risk_plus'].agg(['mean', 'count'])
    rest_trend = rest_trend[rest_trend['count'] >= 10]
    axes[0].bar(rest_trend.index, rest_trend['mean'],
                color=[CLR_HIGH if v > 105 else CLR_LOW for v in rest_trend['mean']])
    axes[0].axhline(100, color='black', ls='--', lw=1.5)
    axes[0].set_xlabel('Days Rest')
    axes[0].set_ylabel('Avg Injury Risk+')
    axes[0].set_title('Risk+ by Days Rest (All Pitchers)', fontweight='bold')
    axes[0].set_xlim(0, 10)

    # 2) Risk+ by days rest, split by role
    for role, grp in df_rest.groupby('role'):
        color = CLR_HIGH if role == 'Starter' else CLR_LOW
        trend = grp.groupby('days_rest')['injury_risk_plus'].mean()
        axes[1].plot(trend.index, trend.values, 'o-', color=color, lw=2, label=role)
    axes[1].axhline(100, color='black', ls='--', lw=1, alpha=0.5)
    axes[1].set_xlabel('Days Rest')
    axes[1].set_ylabel('Avg Injury Risk+')
    axes[1].set_title('Risk+ by Days Rest and Role', fontweight='bold')
    axes[1].legend()
    axes[1].set_xlim(0, 10)

    # 3) Short-rest flag
    df_rest['short_rest'] = df_rest['days_rest'] <= 3
    sr_summary = df_rest.groupby(['role', 'short_rest'])['injury_risk_plus'].mean().unstack()
    sr_summary.plot(kind='bar', ax=axes[2], color=[CLR_LOW, CLR_HIGH])
    axes[2].set_xlabel('Role')
    axes[2].set_ylabel('Avg Injury Risk+')
    axes[2].set_title('Risk+: Normal vs Short Rest by Role', fontweight='bold')
    axes[2].legend(['Normal Rest', 'Short Rest (≤3d)'], fontsize=9)
    axes[2].tick_params(axis='x', rotation=0)
    axes[2].axhline(100, color='black', ls='--', lw=1, alpha=0.5)

    plt.suptitle('Figure 5: Risk+ vs Rest and Recovery', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'risk_vs_rest_days.png')
    plt.show()

---
## Section 7: Risk Interaction Heatmaps

Two-dimensional heatmaps of average predicted Risk+ across feature grids.
These identify **nonlinear danger zones**: combinations that are much riskier
than either factor alone.

Heatmaps produced:
1. Velocity × slider usage
2. Velocity × workload
3. Slider usage × rest days
4. ACWR × prior injury burden
5. Age × workload

In [ ]:
# Risk interaction heatmap helper
def risk_heatmap(df, feat_x, feat_y, label_x, label_y, title, fname, n_bins=10):
    df = df[[feat_x, feat_y, 'injury_risk_plus']].dropna()
    if len(df) < 50:
        print(f'Insufficient data for {title}')
        return
    df['bin_x'] = pd.qcut(df[feat_x], q=n_bins, duplicates='drop')
    df['bin_y'] = pd.qcut(df[feat_y], q=n_bins, duplicates='drop')
    pivot = df.pivot_table(values='injury_risk_plus', index='bin_y', columns='bin_x', aggfunc='mean')
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(pivot, ax=ax, cmap='RdYlBu_r', center=100,
                annot=True, fmt='.0f', linewidths=0.5,
                cbar_kws={'label': 'Avg Injury Risk+'})
    ax.set_xlabel(label_x, fontsize=11)
    ax.set_ylabel(label_y, fontsize=11)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xticklabels([str(i.mid)[:5] for i in pivot.columns], rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels([str(i.mid)[:5] for i in pivot.index], fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / fname)
    plt.show()
    print(f'Saved {fname}')

# 1) Velocity × slider usage
if all(c in analysis.columns for c in ['fb_velo_mean', 'sl_pct']):
    risk_heatmap(analysis, 'fb_velo_mean', 'sl_pct',
                 'Avg FB Velocity (mph)', 'Slider Usage %',
                 'Heatmap: Velocity × Slider Usage',
                 'risk_heatmap_velocity_slider.png')

# 2) Velocity × workload
if all(c in analysis.columns for c in ['fb_velo_mean', 'pitches_28d']):
    risk_heatmap(analysis, 'fb_velo_mean', 'pitches_28d',
                 'Avg FB Velocity (mph)', 'Pitches Last 28 Days',
                 'Heatmap: Velocity × Workload',
                 'risk_heatmap_velocity_workload.png')

# 3) Slider usage × rest days
if all(c in analysis.columns for c in ['sl_pct', 'days_rest']):
    risk_heatmap(analysis, 'sl_pct', 'days_rest',
                 'Slider Usage %', 'Days Rest',
                 'Heatmap: Slider Usage × Rest Days',
                 'risk_heatmap_slider_rest.png')

# 4) ACWR × prior injury burden
if all(c in analysis.columns for c in ['acwr_7_28', 'prior_il_total']):
    risk_heatmap(analysis, 'acwr_7_28', 'prior_il_total',
                 'ACWR (7d / 28d)', 'Prior IL Stints',
                 'Heatmap: ACWR × Prior Injury Burden',
                 'risk_heatmap_acwr_prior_injury.png')

# 5) Age × workload
if all(c in analysis.columns for c in ['age', 'pitches_28d']):
    risk_heatmap(analysis, 'age', 'pitches_28d',
                 'Pitcher Age', 'Pitches Last 28 Days',
                 'Heatmap: Age × Workload',
                 'risk_heatmap_age_workload.png')

---
## Section 8: Pitcher Archetype Risk

Compare Injury Risk+ across pitcher archetypes.

If `pitcher_archetypes.parquet` is not available, derive simple archetypes using rules
based on velocity, pitch mix, workload, and role.

Possible archetypes:
- **Power Starters**: high velocity, high workload
- **Slider-Heavy Starters**: high slider %, high workload
- **Workhorse Starters**: very high pitch counts, above-average velocity
- **Command Starters**: lower velocity, diverse pitch mix
- **High-Leverage Relievers**: short outings, high velocity
- **Bulk Relievers**: moderate workload, varied role
- **Soft-Contact Pitchers**: lower velocity, high movement

**Research questions:**
- Which pitcher types appear most fragile?
- Which types appear relatively durable?
- Are there archetypes that might benefit from role changes?

In [ ]:
# Pitcher archetypes
if archetypes is not None:
    arch_df = analysis.merge(archetypes, on='pitcher', how='left')
    arch_col = [c for c in archetypes.columns if 'archetype' in c.lower() or 'cluster' in c.lower()]
    arch_col = arch_col[0] if arch_col else None
else:
    arch_col = None

# Derive simple rule-based archetypes if none available
if arch_col is None and 'pitch_count' in analysis.columns and 'fb_velo_mean' in analysis.columns:
    arch_df = analysis.copy()
    league_velo = arch_df['fb_velo_mean'].median()

    conditions = [
        (arch_df['pitch_count'] >= 75) & (arch_df['fb_velo_mean'] >= league_velo + 1.5),
        (arch_df['pitch_count'] >= 75) & (arch_df.get('sl_pct', 0) >= 0.28),
        (arch_df['pitch_count'] >= 80),
        (arch_df['pitch_count'] >= 60),
        (arch_df['pitch_count'] < 30) & (arch_df['fb_velo_mean'] >= league_velo + 1.0),
        (arch_df['pitch_count'] < 45),
    ]
    choices = ['Power Starter', 'Slider-Heavy Starter', 'Workhorse Starter',
               'Command Starter', 'High-Leverage Reliever', 'Bulk Reliever']
    arch_df['archetype'] = np.select(conditions, choices, default='Standard')
    arch_col = 'archetype'
    print('Rule-based archetypes derived.')
    print(arch_df[arch_col].value_counts().to_string())
else:
    arch_df = analysis.copy()
    print('Using loaded archetypes.' if arch_col else 'Archetype data unavailable.')

if arch_col and arch_col in arch_df.columns:
    # Average Risk+ by archetype
    arch_summary = (
        arch_df.groupby(arch_col)['injury_risk_plus']
        .agg(['mean', 'median', 'count'])
        .sort_values('mean', ascending=False)
    )

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    colors = [CLR_HIGH if v > 105 else (CLR_LOW if v < 97 else CLR_NEUT)
              for v in arch_summary['mean']]
    axes[0].barh(arch_summary.index[::-1], arch_summary['mean'][::-1], color=colors[::-1])
    axes[0].axvline(100, color='black', ls='--', lw=1.5)
    axes[0].set_xlabel('Average Injury Risk+')
    axes[0].set_title('Average Risk+ by Pitcher Archetype', fontweight='bold')

    if 'injured_next_30d' in arch_df.columns:
        inj_rate = arch_df.groupby(arch_col)['injured_next_30d'].mean() * 100
        inj_rate = inj_rate.reindex(arch_summary.index)
        axes[1].barh(inj_rate.index[::-1], inj_rate.values[::-1],
                     color=[CLR_HIGH if v > inj_rate.mean() else CLR_LOW for v in inj_rate.values[::-1]])
        axes[1].axvline(inj_rate.mean(), color='black', ls='--', lw=1.5)
        axes[1].set_xlabel('Injury Rate (%)')
        axes[1].set_title('30-Day Injury Rate by Archetype', fontweight='bold')

    plt.suptitle('Figure 6: Risk by Pitcher Archetype', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'risk_by_pitcher_archetype.png')
    plt.show()
    print(arch_summary.round(1).to_string())

---
## Section 9: Role-Based Risk

Compare injury risk across pitcher roles: starters, relievers, openers, bulk pitchers,
and hybrid/swingmen.

**Research questions:**
- Is the traditional starter model riskier for some pitcher profiles?
- Are hybrid usage patterns associated with lower risk?
- Are certain pitcher profiles misused in traditional roles?

In [ ]:
# Role-based risk analysis
if 'pitch_count' in analysis.columns:
    df_role = analysis.copy()
    # Role classification from pitch count per outing
    df_role['role'] = pd.cut(
        df_role['pitch_count'],
        bins=[-np.inf, 25, 45, 60, 80, np.inf],
        labels=['Opener/Closer', 'Short Reliever', 'Bulk Reliever', 'Starter', 'Heavy Starter']
    )

    role_risk = df_role.groupby('role', observed=True)['injury_risk_plus'].agg(['mean', 'median', 'count'])
    inj_rate  = df_role.groupby('role', observed=True)['injured_next_30d'].mean() * 100                 if 'injured_next_30d' in df_role.columns else None

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1) Risk+ by role
    colors = [CLR_HIGH if v > 105 else (CLR_LOW if v < 97 else CLR_NEUT) for v in role_risk['mean']]
    axes[0].bar(role_risk.index, role_risk['mean'], color=colors)
    axes[0].axhline(100, color='black', ls='--', lw=1.5)
    axes[0].set_ylabel('Average Injury Risk+')
    axes[0].set_title('Risk+ by Role', fontweight='bold')
    axes[0].tick_params(axis='x', rotation=25)

    # 2) Injury rate by role
    if inj_rate is not None:
        axes[1].bar(inj_rate.index, inj_rate.values,
                    color=[CLR_HIGH if v > inj_rate.mean() else CLR_LOW for v in inj_rate.values])
        axes[1].axhline(inj_rate.mean(), color='black', ls='--', lw=1.5)
        axes[1].set_ylabel('30-Day Injury Rate (%)')
        axes[1].set_title('Injury Rate by Role', fontweight='bold')
        axes[1].tick_params(axis='x', rotation=25)

    # 3) Workload profile by role
    if 'pitches_28d' in df_role.columns:
        workload_by_role = df_role.groupby('role', observed=True)['pitches_28d'].mean()
        axes[2].bar(workload_by_role.index, workload_by_role.values, color=CLR_NEUT, alpha=0.8)
        axes[2].set_ylabel('Avg Pitches Last 28 Days')
        axes[2].set_title('28-Day Workload by Role', fontweight='bold')
        axes[2].tick_params(axis='x', rotation=25)

    plt.suptitle('Figure 7: Risk by Role', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'risk_by_role.png')
    plt.show()
    print(role_risk.round(1).to_string())

---
## Section 10: Performance vs Risk Frontier

Plot the performance-risk tradeoff: pitchers who produce at low risk vs. those
who carry high risk for their performance level.

**Research questions:**
- Which pitchers are efficient from a health-performance perspective?
- Could teams optimize staff construction using both performance and injury risk?

Merge any available performance metrics (WAR, FIP, xFIP, K-BB%, Stuff+) into the analysis.
If none are available, use a statistical proxy from pitch-level data.

In [ ]:
# Performance vs Risk frontier
# Attempt to load FanGraphs or other performance data
perf_path = Path('data/raw/player_metadata/fangraphs_leaderboard.parquet')
perf_proxy_available = perf_path.exists()

if perf_proxy_available:
    perf = pd.read_parquet(perf_path)
    print(f'Performance data loaded: {perf.shape}')
    # TODO: join on player_id / season and create scatter plot
else:
    # Use proxy: average velocity std (inverse = command) as a performance stand-in
    print('No external performance data found.')
    print('Using pitch quality proxy (avg velocity, spin rate) as a performance dimension.')
    if 'fb_velo_mean' in analysis.columns and 'spin_rate_mean' in analysis.columns:
        pitcher_summary = (
            analysis.groupby('pitcher')
            .agg(
                avg_risk_plus   = ('injury_risk_plus', 'mean'),
                avg_velo        = ('fb_velo_mean',     'mean'),
                avg_spin        = ('spin_rate_mean',   'mean'),
                n_appearances   = ('game_date',         'count'),
                player_name     = ('player_name',       'first'),
            )
            .dropna()
        )
        # Performance proxy: velocity × spin (higher = stuff-heavy)
        pitcher_summary['perf_proxy'] = (
            (pitcher_summary['avg_velo'] - pitcher_summary['avg_velo'].mean()) / pitcher_summary['avg_velo'].std()
            + (pitcher_summary['avg_spin'] - pitcher_summary['avg_spin'].mean()) / pitcher_summary['avg_spin'].std()
        )

        fig, ax = plt.subplots(figsize=(10, 7))
        sc = ax.scatter(pitcher_summary['perf_proxy'], pitcher_summary['avg_risk_plus'],
                        alpha=0.6, s=40, c=pitcher_summary['avg_risk_plus'],
                        cmap='RdYlBu_r', vmin=85, vmax=130)
        plt.colorbar(sc, ax=ax, label='Avg Injury Risk+')
        ax.axhline(100, color='black', ls='--', lw=1, alpha=0.5)
        ax.axvline(0, color='black', ls='--', lw=1, alpha=0.5)
        ax.set_xlabel('Stuff Proxy Score (velocity + spin, standardized)')
        ax.set_ylabel('Average Injury Risk+')
        ax.set_title('Performance vs Risk Frontier\n(high-left = high stuff, low risk)', fontweight='bold')

        # Label top/bottom quadrant pitchers
        top_right = pitcher_summary[(pitcher_summary['perf_proxy'] > 0.5) &
                                     (pitcher_summary['avg_risk_plus'] > 105)]
        top_left  = pitcher_summary[(pitcher_summary['perf_proxy'] > 0.5) &
                                     (pitcher_summary['avg_risk_plus'] <= 100)]
        for _, row in top_right.head(5).iterrows():
            ax.annotate(row.get('player_name', str(row.name)),
                        (row['perf_proxy'], row['avg_risk_plus']),
                        fontsize=7, alpha=0.8)

        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'performance_risk_frontier.png')
        plt.show()
    else:
        print('fb_velo_mean or spin_rate_mean not available — skipping performance-risk frontier.')

---
## Section 11: Pre-Injury Risk Trajectory

For pitchers who were eventually placed on the IL, align their observations
relative to the injury date and plot how Injury Risk+ evolves in the weeks prior.

Day 0 = IL placement date.

**Research questions:**
- Does Risk+ rise meaningfully before actual injuries?
- How early does the model begin signaling elevated risk?
- What variables move first: velocity, workload, or slider usage?

In [ ]:
# Pre-injury risk trajectory
if 'injured_next_30d' not in analysis.columns or analysis['injury_risk_plus'].isna().all():
    print('Injury labels or Risk+ scores not available — skipping trajectory analysis.')
else:
    inj_data = pd.read_parquet('data/processed/injuries_clean.parquet')
    inj_data['transaction_date'] = pd.to_datetime(inj_data['transaction_date'])

    # For each IL stint, find pitcher appearances in the 90 days prior
    countdown_rows = []
    for _, stint in inj_data.iterrows():
        pid = stint['player_id']
        inj_date = stint['transaction_date']
        window = analysis[
            (analysis['pitcher'] == pid) &
            (analysis['game_date'] >= inj_date - pd.Timedelta(days=90)) &
            (analysis['game_date'] <  inj_date)
        ].copy()
        if len(window) == 0:
            continue
        window['days_before_injury'] = (inj_date - window['game_date']).dt.days
        countdown_rows.append(window)

    if countdown_rows:
        countdown = pd.concat(countdown_rows, ignore_index=True)
        # Bin by weeks before injury
        countdown['week'] = (countdown['days_before_injury'] // 7) + 1
        countdown = countdown[countdown['week'] <= 12]

        trend_cols = ['injury_risk_plus', 'fb_velo_mean', 'sl_pct', 'pitch_count']
        trend_cols = [c for c in trend_cols if c in countdown.columns]

        fig, axes = plt.subplots(1, len(trend_cols), figsize=(5 * len(trend_cols), 5))
        if len(trend_cols) == 1:
            axes = [axes]

        for ax, col in zip(axes, trend_cols):
            trend = countdown.groupby('week')[col].mean()
            ax.plot(trend.index, trend.values, 'o-', color=CLR_HIGH, lw=2)
            ax.invert_xaxis()
            ax.axvline(1, color='gray', ls=':', alpha=0.5)
            ax.set_xlabel('Weeks Before Injury')
            ax.set_ylabel(col.replace('_', ' ').title())
            ax.set_title(f'{col.replace("_", " ").title()}\nPre-Injury Trend', fontweight='bold')

        plt.suptitle('Figure 8: Pre-Injury Risk Trajectory', fontsize=13, fontweight='bold', y=1.01)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'pre_injury_risk_trajectory.png')
        plt.show()
        print(f'Trajectory built from {len(countdown_rows)} IL stints.')
    else:
        print('No matching appearances found — check that feature_matrix and injury DB share pitcher IDs.')

---
## Section 12: Novel Insight Candidate List

A structured table of candidate baseball insights surfaced by the analysis.

| Column | Description |
|--------|-------------|
| `insight` | The candidate finding |
| `supporting_viz` | Figure or table that shows it |
| `evidence_strength` | weak / moderate / strong |
| `possible_explanation` | Baseball-domain interpretation |
| `follow_up_test` | How to validate or quantify further |
| `predictive_or_causal` | Whether insight is predictive vs. potentially causal |

> All findings are **associative** unless a formal causal design is applied.

In [ ]:
# Novel insight candidate table
INSIGHTS = [
    {
        'insight': 'Slider-heavy, high-velocity pitchers occupy the highest-risk zone — but slider signal is elbow-specific',
        'supporting_viz': 'risk_heatmap_velocity_slider.png',
        'evidence_strength': 'moderate',
        'possible_explanation': 'Slider usage rate (forearm pronation/varus torque) drives UCL stress; '
                                 'not slider velocity (Fleisig 2016 JSES found no velocity difference)',
        'follow_up_test': 'Stratify slider × Risk+ by injury type (elbow vs shoulder vs muscle)',
        'predictive_or_causal': 'predictive',
    },
    {
        'insight': 'Chronic velocity decline (>1 mph below season avg) is associated with elevated Risk+',
        'supporting_viz': 'velo_decline_threshold.png',
        'evidence_strength': 'moderate',
        'possible_explanation': 'Chronic velocity drift captures cumulative fatigue/biomechanical accommodation; '
                                 'distinct from the acute pitch-by-pitch decompensation (>1.5 SD) '
                                 'seen at UCL failure (PMC 12717397 2025)',
        'follow_up_test': 'Compute pitch-by-pitch velo delta to capture acute decompensation signal',
        'predictive_or_causal': 'predictive',
    },
    {
        'insight': 'Short rest appears most harmful for high-workload starters',
        'supporting_viz': 'risk_vs_rest_days.png',
        'evidence_strength': 'moderate',
        'possible_explanation': 'Insufficient recovery time compounds cumulative pitch stress',
        'follow_up_test': 'Simulate short-rest starts in notebook 12 (usage strategy)',
        'predictive_or_causal': 'predictive',
    },
    {
        'insight': 'Certain starters may have lower projected risk in hybrid / bulk roles',
        'supporting_viz': 'risk_by_role.png',
        'evidence_strength': 'weak',
        'possible_explanation': 'Reduced per-outing workload lowers acute stress spikes',
        'follow_up_test': 'Role-transition simulation in notebook 12',
        'predictive_or_causal': 'predictive',
    },
    {
        'insight': 'Risk may accelerate nonlinearly above ACWR 1.3–1.5',
        'supporting_viz': 'risk_vs_workload.png',
        'evidence_strength': 'moderate',
        'possible_explanation': 'ACWR danger zone consistent with team-sport injury literature (Blanch & Gabbett 2016)',
        'follow_up_test': 'Compare actual IL rates below and above ACWR 1.3 with matched controls',
        'predictive_or_causal': 'predictive',
    },
    {
        'insight': 'Power starter archetype shows consistently elevated Risk+',
        'supporting_viz': 'risk_by_pitcher_archetype.png',
        'evidence_strength': 'weak',
        'possible_explanation': 'High velocity combined with heavy workload maximizes cumulative stress',
        'follow_up_test': 'Compare actual IL rates for power starters vs. command starters 2015–2024',
        'predictive_or_causal': 'predictive',
    },
]

insights_df = pd.DataFrame(INSIGHTS)
insights_df.index += 1

out_path = TABLES_DIR / 'baseball_specific_insights_summary.csv'
insights_df.to_csv(out_path, index=True, index_label='rank')
print(f'Saved {out_path}  ({len(insights_df)} insights)')
print()
print(insights_df[['insight', 'evidence_strength', 'predictive_or_causal']].to_string())

---
## Section 13: Conclusions

### What the model learned about pitcher injury risk

The analysis across sections 2-12 surfaced three consistent themes:

**1. Injury history is the dominant signal.**
Prior IL stints (`prior_il_total`) was the top feature by a wide margin (26.8% RF importance), followed by `days_since_last_injury` (14.4%) and `prior_il_days_lost` (12.2%). Workload and pitch-mix features are secondary. A pitcher who has been on the IL before carries substantially higher predicted risk than a workload-matched pitcher with no injury history, regardless of current usage pattern.

**2. Chronic velocity decline shows a clear dose-response.**
Section 4b found that pitchers more than 2 mph below their season velocity average carried a mean Risk+ of ~119 vs ~97 for pitchers at or above their season average. Actual 30-day injury rates tracked the same pattern (~12% vs ~6%). Multi-outing velocity drift, distinct from the acute pitch-by-pitch decompensation documented in PMC 12717397 (2025), is a real predictive signal captured by `velo_delta_vs_season`.

**3. Cumulative workload matters more than acute-to-chronic ratios.**
Interventional SHAP analysis (NB10) confirmed that `pitches_90d` genuinely outranks `acwr_7_28` in feature importance, not a path-dependent artifact. The 90-day cumulative load appears to capture injury risk more reliably than the ACWR ratio in this dataset, consistent with recent meta-analyses questioning ACWR's universal applicability (Bowen et al. 2020).

---

### Patterns that were expected

- Injury history dominance (consistent with all published recurrent-injury literature)
- Slider usage associated with elevated Risk+ (Tanaka 2024; mechanism is usage rate, not velocity, per Fleisig 2016)
- Power starters showing the highest archetype-level risk
- Starters carrying higher average Risk+ than relievers (consistent with Hazard of Arm Injury, PMC 2022)
- ACWR risk acceleration above 1.3-1.5 (Blanch & Gabbett 2016 danger zone)

### Patterns that were surprising

- **Injury history overshadowed workload by a larger margin than expected.** The model is largely distinguishing previously-injured pitchers from healthy ones, not quantifying workload-driven risk within the healthy population.
- **ACWR underperformed absolute cumulative load.** Despite being the standard workload metric in sports science, `acwr_7_28` ranked ~48th in interventional SHAP (vs `pitches_90d` at ~5th). Within the MLB pitcher population, chronic load accumulation appears to matter more than acute spikes relative to chronic baseline.
- **Pitch count simulation showed inverted risk (NB12).** Higher pitch counts in a start were associated with *lower* predicted risk, a survivorship bias artifact: managers only allow healthy pitchers deep into games, so the training data encodes "high pitch count ↔ healthy" rather than a causal dose-response.

---

### Key limitations

- **Predicts all injury types combined.** The model conflates UCL tears, shoulder impingement, oblique strains, and hamstring injuries into a single binary label. Slider usage → elbow injury specificity is diluted; a UCL-specific model would sharpen that signal considerably.
- **Associative, not causal.** All patterns reflect what the model learned from observational data. Short rest, slider usage, and role type are all confounded by manager decision-making and pitcher health signals not captured in the feature set.
- **Modest absolute performance.** Binary classifier PR-AUC = 0.134 (vs 0.099 naive baseline); survival model C-index = 0.566 (vs 0.514 random). Both beat their baselines but remain limited. Public Statcast data captures pitch-level mechanics at population resolution; injury-specific signals (e.g., UCL torque, individual biomechanical load) require instrumented data not publicly available.

---

### Insights tested in Notebook 12

1. **Pitch count reduction:** additive perturbation (corrects for proportional-scaling bug that flattened the curve); survivorship bias caveat added; response is modest within non-injured population
2. **Injury recency simulation:** `days_since_last_injury` shows a nearly flat simulated response when `prior_il_total` is held fixed. Injury count, not recency, drives risk
3. **ACWR perturbation:** moderate response above ACWR 1.3, consistent with Section 5

### Findings NOT to treat as causal

- Slider usage → elbow injury (confounded by pitcher type and usage context)
- Short rest → injury (confounded by manager selection of healthy pitchers for short-rest starts)
- Role type → injury rate (confounded by pitcher selection into roles)
- High pitch count → low risk (survivorship bias; see NB12 §3)

> **Next step:** Notebook 12: Usage Strategy Simulation  
> Test the workload and pitch mix interventions suggested here using counterfactual model inference.